# 05 · GRPO (Group Relative Policy Optimization) 原理与实战

> 前置:先跑通 [`00`](00_环境准备与GPU检测.ipynb)。本章**不需要**奖励模型,用一个**自定义奖励函数**即可,单张 16GB 可跑。

GRPO 是 DeepSeek 提出、近来在**推理模型(如 DeepSeek-R1)**上大放异彩的方法,可以看作 **PPO 的「瘦身版」**。
它特别适合**答案能被自动验证**的任务 —— 比如数学题(答案对不对一目了然)、代码(能不能通过测试)、格式(是否符合要求)。
本章我们用真实的小学数学应用题数据集 **GSM8K**(`openai/gsm8k`)来体会它 —— 这正是训练推理模型最经典的数据之一。

## 一、GRPO 相比 PPO 省掉了什么

PPO 需要一个**价值网络 (critic)** 来估计基线、算优势 —— 这既占显存又难训。
GRPO 的洞见是:**干嘛非要学一个价值网络?对同一个问题多采样几个回答,用「组内的平均分」当基线不就行了?**

具体做法:对每个 prompt,采样一**组** \(G\) 个回答 \(\{y_1, ..., y_G\}\),各自打分得到 \(\{r_1, ..., r_G\}\),
然后用**组内标准化**得到每个回答的优势:

\[ A_i = \frac{r_i - \mathrm{mean}(r_1,...,r_G)}{\mathrm{std}(r_1,...,r_G)} \]

直觉:**在这一组里比平均好的回答,优势为正(被鼓励);比平均差的,优势为负(被抑制)。**
于是:

- ❌ 不需要价值网络(省一个模型、更稳);
- ✅ 仍保留 PPO 的裁剪和对参考模型的 KL 惩罚;
- ✅ 奖励可以是**任意一段代码/规则**(不必是可导的神经网络),这对「可验证任务」极其自然。

```mermaid
flowchart LR
    p["一个 prompt"] --> g["采样一组 G 个回答"]
    g --> r["逐个用奖励函数打分<br/>(可以是任意规则/代码)"]
    r --> norm["组内标准化<br/>A = (r - mean) / std"]
    norm --> upd["裁剪更新策略 + KL 惩罚"]
    upd --> p
```

## 二、准备可验证任务的数据(优先用 HuggingFace 的 GSM8K)

我们**优先从 HuggingFace 拉取** [`openai/gsm8k`](https://huggingface.co/datasets/openai/gsm8k)(`main` 配置):它是小学数学应用题,
每条有 `question` 和 `answer`,其中 `answer` 末尾用 `#### 数字` 给出标准答案 —— 天然「可验证」。下载失败则回退本地 `data/prompts.jsonl` 里 `task=math` 的题目。

- GRPO 数据只需 `prompt` 列;其余列(如 `answer`)会被**自动透传**给奖励函数当参数。
- 我们把标准答案抽成纯数字字符串 `answer`,并加一个系统提示要求「先推理、最后给出数字答案」。

> GSM8K 需要**多步推理**,所以后面生成长度(`max_completion_length`)会比纯算术题设得更大。

In [ ]:
import os
# os.environ["HF_ENDPOINT"] = "https://hf-mirror.com"
from datasets import load_dataset

SYS = "你是一个数学助手。请一步步推理,并在最后一行用『答案:<数字>』给出最终的整数答案。"
N_SAMPLES = 128    # 演示规模:数学题只取一小部分

def load_math_dataset():
    """优先用 HuggingFace 经典数学数据集 openai/gsm8k(main);失败回退本地 prompts.jsonl(task=math)。
    统一整理成 GRPO 需要的对话格式,并透传标准答案 answer(纯数字字符串)。"""
    try:
        ds = load_dataset("openai/gsm8k", "main", split="train")
        def conv(ex):
            ans = ex["answer"].split("####")[-1].strip().replace(",", "")
            return {"prompt": [{"role": "system", "content": SYS},
                               {"role": "user", "content": ex["question"]}],
                    "answer": ans}
        ds = ds.map(conv, remove_columns=ds.column_names)
        ds = ds.shuffle(seed=42).select(range(min(N_SAMPLES, len(ds))))
        print(f"✅ 已从 HuggingFace 加载 openai/gsm8k (main),取 {len(ds)} 条")
        return ds
    except Exception as e:
        print(f"⚠️ 从 HF 下载失败({type(e).__name__}),回退本地 data/prompts.jsonl\n   {e}")
        raw = load_dataset("json", data_files="../data/prompts.jsonl", split="train")
        raw = raw.filter(lambda x: x["task"] == "math")
        def conv(ex):
            return {"prompt": [{"role": "system", "content": SYS},
                               {"role": "user", "content": ex["prompt"]}],
                    "answer": str(ex["answer"])}
        return raw.map(conv, remove_columns=raw.column_names)

grpo_ds = load_math_dataset()
print(grpo_ds)
print(grpo_ds[0])

## 三、写奖励函数:这是 GRPO 的灵魂

GRPO 的奖励函数签名约定:`func(completions, **kwargs)`,返回一个 `list[float]`(每个 completion 一个分数)。
数据集里的额外列(这里是 `answer`)会作为**关键字参数**传进来。用 `**kwargs` 兜住最保险。

我们写两个奖励函数(GRPO 会把它们**相加**):

1. `reward_correct`:回答里最后一个数字 == GSM8K 标准答案就得 1 分,否则 0 分 —— 这是**可验证奖励**的核心。
2. `reward_has_answer`:轻微的**塑形奖励 (reward shaping)**,只要回答里给出了数字就加一点分,鼓励模型「给出明确答案」。

> 注意:GSM8K 需要多步推理,所以我们**不**再奖励「简短」(那会抑制推理链);只奖励「答对」和「有明确答案」。

In [ ]:
import re

def _text(completion):
    # 兼容对话格式(list of dict)与纯字符串
    if isinstance(completion, str):
        return completion
    if isinstance(completion, list) and completion and isinstance(completion[0], dict):
        return completion[0]["content"]
    return str(completion)

def _last_int(text):
    """取文本里最后一个整数(先去掉千分位逗号)。"""
    nums = re.findall(r"-?\d+", text.replace(",", ""))
    return nums[-1] if nums else None

def reward_correct(completions, answer, **kwargs):
    """核心可验证奖励:回答里最后一个数字 == 标准答案则得 1 分。"""
    return [1.0 if _last_int(_text(c)) == str(a) else 0.0
            for c, a in zip(completions, answer)]

def reward_has_answer(completions, **kwargs):
    """塑形奖励:回答里至少出现一个数字就给一点分,鼓励『给出明确答案』(不惩罚推理过程)。"""
    return [0.1 if _last_int(_text(c)) is not None else 0.0 for c in completions]

# 快速自测
demo = ["我们一步步算:6 乘以 7 = 42,所以 答案:42", "大概是 50 吧", "不知道"]
print("reward_correct   :", reward_correct(demo, answer=["42", "17", "3"]))
print("reward_has_answer:", reward_has_answer(demo))

## 四、配置 GRPOConfig 并训练

关键超参:

- `num_generations`:每个 prompt 采样一组多少个回答(就是公式里的 \(G\),GRPO 的核心)。
- `per_device_train_batch_size`:必须能被 `num_generations` 整除(下面 8 / 4 = 每步 2 个 prompt)。
- `max_completion_length`:每个回答最多生成多少 token。**GSM8K 要推理,所以设得比纯算术题大(这里 200)。**
- 用 LoRA(`peft_config`)让策略只训练少量参数,省显存。

> 生成变长 + 一组多份采样,是 GRPO 比较吃时间的地方;演示规模下跑几十步即可看到 reward 变化。

In [ ]:
from trl import GRPOConfig, GRPOTrainer
from peft import LoraConfig

MODEL_NAME = "Qwen/Qwen2.5-0.5B-Instruct"

lora_config = LoraConfig(
    task_type="CAUSAL_LM",
    r=16, lora_alpha=32, lora_dropout=0.05,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj"],
)

grpo_args = GRPOConfig(
    output_dir="../outputs/grpo-qwen0.5b",
    per_device_train_batch_size=8,     # 需能被 num_generations 整除(8 / 4 = 2 个 prompt/步)
    gradient_accumulation_steps=1,
    num_generations=4,                 # 每题采样一组 4 个回答
    max_prompt_length=256,
    max_completion_length=200,         # GSM8K 要推理,留足生成长度
    learning_rate=1e-5,
    max_steps=30,                      # 玩具规模;想要明显效果调大
    temperature=0.9,                   # 采样要有多样性,组内才有区分
    logging_steps=1,
    report_to=[],
    bf16=True,
)

trainer = GRPOTrainer(
    model=MODEL_NAME,
    reward_funcs=[reward_correct, reward_has_answer],
    args=grpo_args,
    train_dataset=grpo_ds,
    peft_config=lora_config,
)
trainer.train()
trainer.save_model("../outputs/grpo-qwen0.5b")
print("GRPO 训练完成,适配器已保存。")

## 五、看奖励曲线 + 对齐前后答题对比

`GRPOTrainer` 会把 `reward`、`kl`、`completion_length` 等写进 `state.log_history`。

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

hist = pd.DataFrame(trainer.state.log_history)
print("记录到的指标列:", list(hist.columns))
cols = [c for c in ["reward", "reward_std", "kl", "completion_length"] if c in hist.columns]
if cols:
    hist[cols].plot(subplots=True, figsize=(9, 2.2 * len(cols)), marker="o",
                    title="GRPO 训练指标(reward 应逐步上升)")
    plt.tight_layout(); plt.show()
else:
    print("未找到常见指标列,请查看上面打印的列名。")

In [ ]:
import torch

tok = trainer.processing_class
policy = trainer.model
policy.eval()

@torch.no_grad()
def solve(question, max_new_tokens=200):
    msgs = [{"role": "system", "content": SYS}, {"role": "user", "content": question}]
    ids = tok.apply_chat_template(msgs, add_generation_prompt=True, return_tensors="pt").to(policy.device)
    out = policy.generate(ids, max_new_tokens=max_new_tokens, do_sample=False,
                          pad_token_id=tok.eos_token_id)
    return tok.decode(out[0][ids.shape[1]:], skip_special_tokens=True).strip()

# 直接用训练集里的真实 GSM8K 题目看效果
for i in range(2):
    q = grpo_ds[i]["prompt"][-1]["content"]
    print("=" * 72)
    print(f"问题: {q}")
    print(f"标准答案: {grpo_ds[i]['answer']}")
    print(f"模型答:\n{solve(q)}")

## 小结

- GRPO = **PPO 去掉价值网络**,改用「**同一 prompt 采样一组回答,组内标准化当优势**」。
- 奖励可以是**任意规则/代码**(答案对不对、格式合不合规),不必是神经网络 → 天然适配**可验证任务**。
- 奖励函数签名 `func(completions, **kwargs)`,数据集额外列自动透传;多个奖励函数会相加。
- 这正是 DeepSeek-R1 等**推理模型**训练的核心思路之一。

下一站:[`06_对齐前后对比与评估`](06_对齐前后对比与评估.ipynb) —— 用 02 的奖励模型当裁判,系统对比对齐效果。